# Energy Reconstruction Using CNN - Both Charges and Cos(Zenith)

In [1]:
import numpy as np
import tensorflow as tf
import os
import sys
import matplotlib.pyplot as plt

from datetime import datetime

from tensorflow import keras
from keras import layers
from keras import models
from keras import callbacks
from keras import backend

#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Conv2D, Flatten
#from tensorflow.keras.callbacks import ModelCheckpoint

from data_tools import load_preprocessed, dataPrep, nameModel
from hex_filter import hex_filter, MaskedConv2D, hex_init

simPrefix = os.getcwd()+'\\simdata'

In [3]:
tf.__version__

'2.8.0'

## Model Design

In [4]:
# Name for model
name = 'hex'
i = 0
while(os.path.exists('untrainedModels/{}.h5'.format(name+str(i)))):
    i = i + 1
name = name+str(i)

# Data preparation: no merging of charge (q), no time layers included (t=False), data normalized from 0-1
prep = {'q':None, 't':False, 'normed':True, 'reco':'plane', 'cosz':True}
print(name)

hex1


In [5]:
# Create model using functional API for multiple inputs
charge_input=keras.Input(shape=(10,10,2,))

#x = layers.Conv2D(filters=1, 
#                      kernel_size = 3,
#                      kernel_initializer=hex_filter,
#                      strides=2, 
#                      padding='valid') (charge_input)

#conv1_layer = layers.Conv2D(64,kernel_size=3,padding='same',activation='relu')(charge_input)
#conv1_layer = layers.Conv2D(filters=64,kernel_size=3,kernel_initializer=hex_filter,padding='same',activation='relu')(charge_input)
conv1_layer = MaskedConv2D(filters=64,kernel_size=3,padding='same',activation='relu',kernel_initializer=hex_init)(charge_input)
#conv1_layer = MaskedConv2D(num_outputs=64,padding='same')(charge_input)
#activ1_layer = layers.Activation('relu')(conv1_layer)

#conv2_layer = layers.Conv2D(32,kernel_size=3,padding='same',activation='relu')(conv1_layer)
#conv2_layer = layers.Conv2D(32,kernel_size=3,kernel_initializer=hex_filter,padding='same',activation='relu')(conv1_layer)
conv2_layer = MaskedConv2D(32,kernel_size=3,padding='same',activation='relu',kernel_initializer=hex_init)(conv1_layer)
#conv2_layer = MaskedConv2D(num_outputs=32,padding='same')(activ1_layer)
#activ2_layer = layers.Activation('relu')(conv2_layer)

#conv3_layer = layers.Conv2D(16, kernel_size=3, padding='same',activation="relu")(conv2_layer)
#conv3_layer = layers.Conv2D(16, kernel_size=3,kernel_initializer=hex_filter, padding='same',activation="relu")(conv2_layer)
conv3_layer = MaskedConv2D(16, kernel_size=3, padding='same',activation="relu",kernel_initializer=hex_init)(conv2_layer)
#conv3_layer = MaskedConv2D(num_outputs=16,padding='same')(activ2_layer)
#activ3_layer = layers.Activation('relu')(conv3_layer)

flat_layer = layers.Flatten()(conv3_layer)
#flat_layer = layers.Flatten()(activ3_layer)

#flat_layer = layers.Flatten()(conv3_layer)
zenith_input=keras.Input(shape=(1,))
concat_layer = layers.Concatenate()([flat_layer,zenith_input])

dense1_layer = layers.Dense(256,activation='relu')(concat_layer)

dense2_layer = layers.Dense(256,activation='relu')(dense1_layer)

dense3_layer = layers.Dense(256,activation="relu")(dense2_layer)

output = layers.Dense(1)(dense3_layer)

model = models.Model(inputs=[charge_input,zenith_input],outputs=output,name=name)

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

TypeError: ('Keyword argument not understood:', 'kernel_initalizer')

In [ ]:
model.summary()

In [ ]:
model.save('untrainedModels/%s.h5'% name)
np.save('untrainedModels/%s.npy' % name,prep)